# DeepGuard — DF40 official download to Google Drive (v3)

The official DF40 testing folder is public. This notebook first enumerates its files with modern `gdown --json`, then downloads files individually into the mounted Drive. This avoids the folder-download failure seen with the previous approach.


In [ ]:
!pip -q install -U 'gdown>=6.1.0'


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, os, shutil

drive.mount('/content/drive', force_remount=False)
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DEST=ROOT/'datasets/DF40'
DEST.mkdir(parents=True, exist_ok=True)
DF40_URL='https://drive.google.com/drive/folders/1U8meBbqVvmUkc5GD0jxct6xe6Gwk9wKD?usp=drive_link'
print('Destination:', DEST)
print('Free Drive space (GiB):', round(shutil.disk_usage('/content/drive').free/1024**3,1))


In [ ]:
# Enumerate first — no data are downloaded by this cell.
cmd=[sys.executable,'-m','gdown','--folder',DF40_URL,'--json']
print('Resolving official DF40 folder contents...')
r=subprocess.run(cmd,capture_output=True,text=True)
print(r.stderr[-2000:])
if r.returncode!=0:
    raise RuntimeError('gdown enumeration failed: '+r.stderr[-2000:])
raw=r.stdout.strip()
start=raw.find('[')
if start<0:
    raise RuntimeError('No JSON file list returned. Full output:\n'+raw[-4000:])
files=json.loads(raw[start:])
print('Files resolved:',len(files))
print('First 10:')
for x in files[:10]: print(x.get('path'))
# Store manifest on Drive so the run is auditable/resumable.
(ROOT/'manifests').mkdir(parents=True,exist_ok=True)
(ROOT/'manifests/df40_gdrive_file_manifest.json').write_text(json.dumps(files,indent=2))


In [ ]:
# Download each resolved file directly to Drive. Existing files of the same size are skipped.
from urllib.parse import unquote
from pathlib import PurePosixPath

completed=0
skipped=0
failed=[]
for i,item in enumerate(files,1):
    rel=unquote(item.get('path','')).lstrip('/')
    if not rel: continue
    target=DEST/rel
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>0:
        skipped+=1
        if i%100==0: print(f'[{i}/{len(files)}] existing: {rel}')
        continue
    url=item.get('url')
    if not url:
        failed.append((rel,'missing URL')); continue
    print(f'[{i}/{len(files)}] downloading: {rel}')
    try:
        rr=subprocess.run([sys.executable,'-m','gdown',url,'-O',str(target),'--continue'],text=True)
        if rr.returncode!=0 or not target.exists() or target.stat().st_size==0:
            failed.append((rel,f'exit={rr.returncode}'))
        else:
            completed+=1
    except Exception as e:
        failed.append((rel,repr(e)))
    # Persist progress after every file.
    (ROOT/'manifests/df40_download_progress.json').write_text(json.dumps({'total':len(files),'completed':completed,'skipped':skipped,'failed':failed},indent=2))

print('DONE')
print('New:',completed,'Existing:',skipped,'Failed:',len(failed))
if failed: print('First failures:',failed[:10])


In [ ]:
# Final inventory
files_local=[p for p in DEST.rglob('*') if p.is_file()]
size=sum(p.stat().st_size for p in files_local)
print('Local files:',len(files_local))
print('Size (GiB):',round(size/1024**3,2))
print('Destination:',DEST)
if len(files_local)==0: raise RuntimeError('No DF40 files were downloaded.')
